# 04 — Train: Ridge regression

Searches Ridge regularization for the configured target station's direct 24-hour water-level forecast over the joined feature artifacts, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract  
**Outputs:** in-notebook prediction preview/test metrics, an MLflow run hierarchy, and the selected model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. Its predictor columns are the source of truth for the model inputs: target-station engineered features plus raw measurements from every retained station at issue time `t`. The Ridge alpha search and validation policy are explicit constants so every fold and MLflow run remains inspectable.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `FULL_FEATURE_COLUMNS` | all metadata-declared predictors | The complete predictor contract used for common eligibility; raw timestamps and metadata fields are not model inputs. |
| `FEATURE_SUBSETS` | six predefined subsets | Candidate feature lists derived from the full metadata contract in metadata order. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` | The 24 future water levels predicted directly from one issue-time feature vector. |
| `FORECAST_HORIZON_HOURS` | `24` | Number of direct future target outputs and the metadata contract width. |
| `RIDGE_ALPHAS` | `[0.01, 0.1, 1.0, 10.0, 100.0]` | Candidate L2 regularization strengths. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"mae"` | Aggregate CV metric used to select alpha; `"rmse"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"ridge"` | Experiment receiving the parent, nested fold, and final test runs. |
| `MODEL_PATH` | `models/ridge_{TARGET_STATION_ID}.joblib` | Bundled scaler and selected Ridge estimator trained on all eligible training rows. |
| `MODEL_METADATA_PATH` | `models/ridge_{TARGET_STATION_ID}.json` | Reproducibility manifest containing the feature contract, selected subset and alpha, CV results, and training range. |

## Joint feature-subset and alpha search

The notebook compares one global `(feature subset, alpha)` pair with five expanding-window folds. The six predefined subsets are derived from metadata-declared predictors and retain their metadata order:

| Subset | Intended predictors | Current size |
| --- | --- | ---: |
| `full` | All declared predictors | 81 |
| `all_station_hydrology_quality_time` | Water-level history, imputation indicators, and calendar signals for every station; excludes weather | 55 |
| `raw_all_stations` | Current `water_level`, `imputed`, precipitation, and temperature for every station | 32 |
| `target_station_full` | All declared predictors for the target station only | 53 |
| `target_station_hydrology_quality_time` | Target-station water-level history, imputation indicators, and calendar signals | 41 |
| `current_water_levels_all_stations` | Current `water_level` for every station | 8 |

The full contract determines eligibility once for both artifacts. Consequently, all candidates use the same 48,403 training rows, 15,196 sealed-test rows, and identical fold indices in the current data. A missing predictor excluded by a candidate still removes that timestamp for every candidate; smaller subsets therefore do not gain additional eligible rows in this controlled ablation. The search performs `6 × 5 × 5 = 150` fold fits, then retrains only the selected candidate and evaluates the sealed test once.

In [ ]:
import hashlib
import json
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.metrics import metric_tables
from src.training import (
    absolute_error_boxplot_payload,
    build_feature_subsets,
    cv_error_boxplot_payload,
    error_boxplots_figure,
    load_joined_training_data,
    numeric_predictors,
    predicted_vs_actual_figure,
    prediction_preview,
    prepare_model_rows,
    summarize_cv_metrics,
    time_series_splits,
    validate_predictions,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
MLFLOW_EXPERIMENT_NAME = "ridge"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
station_id = TARGET_STATION_ID
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"ridge_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"ridge_{station_id}.json"

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts. One row is one timestamp `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true, and all 24 `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` values are present.
2. **Every model input is present.** All full-contract predictors must be available: the target station's engineered features plus every retained station's raw water level, imputation flag, precipitation, and temperature at issue time `t`.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it. Eligibility is deliberately based on `FULL_FEATURE_COLUMNS`, not a candidate subset, so all 30 candidates compare the same cohort.

## Shared helpers

Joined-contract loading, common-cohort preparation, ordered feature subsets, chronological folds, prediction checks, metric summaries, previews, and evaluation figures come from `src.training`. Ridge candidate ranking remains local because its subset/alpha tie-breaking policy is estimator-specific.

In [ ]:
def select_candidate(
    cv_results: pd.DataFrame, metric: str = "mae"
) -> tuple[str, float]:
    """Select one subset/alpha pair with deterministic tie-breaking."""
    if metric not in {"mae", "rmse"}:
        raise ValueError("CV selection metric must be either 'mae' or 'rmse'")
    metric_column = f"{metric}_mean"
    required_columns = {"subset", "alpha", "feature_count", metric_column}
    missing = sorted(required_columns.difference(cv_results.columns))
    if missing or cv_results.empty:
        raise ValueError(f"CV results are empty or missing columns: {missing}")
    if cv_results[[metric_column, "alpha", "feature_count"]].isna().any().any():
        raise ValueError("CV candidate results contain null ranking values")
    ranked = cv_results.sort_values(
        [metric_column, "feature_count", "alpha", "subset"],
        kind="stable",
    )
    winner = ranked.iloc[0]
    return str(winner["subset"]), float(winner["alpha"])

## Load joined feature artifacts

Loads the joined feature metadata, `all_stations_train_features.parquet`, and `all_stations_test_features.parquet` from the Stage-3 directory. Their station, horizon, and column contracts are checked before the fit, so a missing or incompatible artifact fails before any model work begins.

In [ ]:
contract, train_features, test_features = load_joined_training_data(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
)
TARGET_COLUMNS = list(contract.target_columns)
FULL_FEATURE_COLUMNS = list(contract.predictor_columns)
FEATURE_SUBSETS = build_feature_subsets(
    contract,
    weather_variables=WEATHER_VARIABLES,
)
INPUT_PARQUET_SHA256_PARAMS = {
    "train_input_sha256": _sha256_file(train_path),
    "test_input_sha256": _sha256_file(test_path),
}

## Apply the eligibility cohort

Prepares the train and test cohorts independently with `prepare_model_rows()`. Each cohort keeps only target-valid rows with complete predictors and targets, then sorts them chronologically. If either split has no eligible row, the notebook stops rather than fitting on an empty frame or reporting a metric computed from nothing.

In [ ]:
train_rows = prepare_model_rows(
    train_features,
    contract,
    artifact_name="train",
)
test_rows = prepare_model_rows(
    test_features,
    contract,
    artifact_name="test",
)

## Joint time-series subset and alpha search

Eligible training rows are sorted by issue time before `TimeSeriesSplit` creates five expanding-window folds. The explicit `test_size` allocates the post-initial-training portion across the folds, while the 24-row gap acts as the requested hourly embargo. Each `(subset, alpha)` candidate has one MLflow parent and each fold has one nested child run: `6 × 5 × 5 = 150` fits. The current execution must produce the complete 30-candidate Cartesian product before selection.

Every fold fits its own `StandardScaler` and 24-output `Ridge` model using only that fold's training rows and the candidate's explicit columns. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splitter, cv_splits, validation_test_size = time_series_splits(
    len(train_rows),
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
expected_candidate_keys = {
    (subset_name, float(alpha))
    for subset_name in FEATURE_SUBSETS
    for alpha in RIDGE_ALPHAS
}

for subset_name, feature_columns in FEATURE_SUBSETS.items():
    for alpha in RIDGE_ALPHAS:
        fold_aggregate_rows = []
        fold_horizon_rows = []
        with mlflow.start_run(
            run_name=f"ridge_cv_{subset_name}_{alpha:g}",
            nested=False,
            tags={
                "phase": "cv",
                "run_type": "candidate_parent",
                "subset": subset_name,
                "execution_uuid": NOTEBOOK_EXECUTION_UUID,
            },
        ):
            mlflow.log_params(
                {
                    "phase": "cv",
                    "run_type": "candidate_parent",
                    "subset": subset_name,
                    "feature_count": len(feature_columns),
                    "feature_columns": json.dumps(feature_columns),
                    **INPUT_PARQUET_SHA256_PARAMS,
                    "alpha": alpha,
                    "n_validation_folds": N_VALIDATION_FOLDS,
                    "validation_test_size": validation_test_size,
                    "embargo_hours": EMBARGO_HOURS,
                    "selection_metric": CV_SELECTION_METRIC,
                    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                    "common_train_rows": len(train_rows),
                }
            )

            for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
                cv_splits, start=1
            ):
                fold_train_rows = train_rows.iloc[fold_train_indices]
                fold_validation_rows = train_rows.iloc[fold_validation_indices]
                fold_scaler = StandardScaler()
                fold_train_predictors = fold_scaler.fit_transform(
                    numeric_predictors(fold_train_rows, feature_columns)
                )
                fold_validation_predictors = fold_scaler.transform(
                    numeric_predictors(fold_validation_rows, feature_columns)
                )
                fold_ridge = Ridge(alpha=alpha)
                fold_ridge.fit(fold_train_predictors, fold_train_rows[TARGET_COLUMNS])
                fold_predictions = validate_predictions(
                    fold_ridge.predict(fold_validation_predictors),
                    expected_rows=len(fold_validation_rows),
                    target_columns=TARGET_COLUMNS,
                    artifact_name="fold",
                )

                fold_aggregate, fold_per_horizon = metric_tables(
                    fold_validation_rows[TARGET_COLUMNS],
                    fold_predictions,
                    target_columns=TARGET_COLUMNS,
                    station_id=station_id,
                )
                fold_aggregate_rows.append(fold_aggregate.iloc[0])
                fold_horizon_rows.append(fold_per_horizon)
                with mlflow.start_run(
                    run_name=f"ridge_cv_{subset_name}_{alpha:g}_fold_{fold_number}",
                    nested=True,
                    tags={
                        "phase": "cv",
                        "run_type": "fold",
                        "subset": subset_name,
                        "fold": str(fold_number),
                        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                    },
                ):
                    mlflow.log_params(
                        {
                            "phase": "cv",
                            "run_type": "fold",
                            "subset": subset_name,
                            "feature_count": len(feature_columns),
                            **INPUT_PARQUET_SHA256_PARAMS,
                            "alpha": alpha,
                            "fold": fold_number,
                            "train_rows": len(fold_train_rows),
                            "validation_rows": len(fold_validation_rows),
                            "gap_rows": EMBARGO_HOURS,
                            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                            "train_start": fold_train_rows["timestamp"]
                            .iloc[0]
                            .isoformat(),
                            "train_end": fold_train_rows["timestamp"]
                            .iloc[-1]
                            .isoformat(),
                            "validation_start": fold_validation_rows["timestamp"]
                            .iloc[0]
                            .isoformat(),
                            "validation_end": fold_validation_rows["timestamp"]
                            .iloc[-1]
                            .isoformat(),
                            "train_index_start": int(fold_train_indices[0]),
                            "train_index_end": int(fold_train_indices[-1]),
                            "validation_index_start": int(fold_validation_indices[0]),
                            "validation_index_end": int(fold_validation_indices[-1]),
                        }
                    )
                    mlflow.log_metrics(
                        {
                            "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                            "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                            **{
                                f"fold_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                                for row in fold_per_horizon.itertuples()
                            },
                        }
                    )

            fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
            fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
            parent_metrics = summarize_cv_metrics(
                fold_aggregate_metrics,
                fold_horizon_metrics,
            )
            candidate_key = (subset_name, float(alpha))
            cv_horizon_rows_by_candidate[candidate_key] = fold_horizon_rows.copy()
            mlflow.log_metrics(parent_metrics)
            cv_results_rows.append(
                {
                    "subset": subset_name,
                    "feature_count": len(feature_columns),
                    "alpha": float(alpha),
                    "mae_mean": parent_metrics["cv_mae_mean"],
                    "mae_std": parent_metrics["cv_mae_std"],
                    "rmse_mean": parent_metrics["cv_rmse_mean"],
                    "rmse_std": parent_metrics["cv_rmse_std"],
                    **{
                        metric_name: metric_value
                        for metric_name, metric_value in parent_metrics.items()
                        if metric_name
                        not in {
                            "cv_mae_mean",
                            "cv_mae_std",
                            "cv_rmse_mean",
                            "cv_rmse_std",
                        }
                    },
                }
            )

cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'candidate_parent'"
    ),
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'fold'"
    ),
)
parent_keys = {
    (str(row["tags.subset"]), float(row["params.alpha"]))
    for _, row in current_cv_runs.iterrows()
}
if (
    len(current_cv_runs) != len(expected_candidate_keys)
    or parent_keys != expected_candidate_keys
):
    raise ValueError(
        "Current execution must produce the complete 30-candidate subset/alpha product: "
        f"expected {len(expected_candidate_keys)} {sorted(expected_candidate_keys)}, "
        f"got {len(current_cv_runs)} {sorted(parent_keys)}"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, got {len(current_fold_runs)}"
    )
fold_keys = {
    (
        str(row["tags.subset"]),
        float(row["params.alpha"]),
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (subset_name, float(alpha), fold_number)
    for subset_name, alpha in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if fold_keys != expected_fold_keys:
    raise ValueError(
        "Current execution fold runs do not cover every candidate and fold"
    )
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
if set(zip(cv_results["subset"], cv_results["alpha"])) != expected_candidate_keys:
    raise ValueError(
        "The in-memory CV result table does not match the candidate product"
    )
cv_results = cv_results.sort_values(["subset", "alpha"], kind="stable").reset_index(
    drop=True
)
selected_subset, selected_alpha = select_candidate(cv_results, CV_SELECTION_METRIC)
selected_feature_columns = FEATURE_SUBSETS[selected_subset]
fold_horizon_rows = cv_horizon_rows_by_candidate[(selected_subset, selected_alpha)]
print(
    f"Selected Ridge candidate by CV {CV_SELECTION_METRIC.upper()}: "
    f"{selected_subset!r}, alpha={selected_alpha:g}"
)
display(
    cv_results[
        [
            "subset",
            "feature_count",
            "alpha",
            "mae_mean",
            "mae_std",
            "rmse_mean",
            "rmse_std",
        ]
    ]
)

## Retrain the selected subset and alpha

The selected `(feature subset, alpha)` pair is retrained once on all eligible, chronologically ordered training rows. The scaler and Ridge estimator are bundled in a single pipeline, fitted on the full eligible training cohort, and persisted with a reproducibility manifest before the sealed test predictors are scored.

In [ ]:
final_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=selected_alpha)),
    ]
)
final_model.fit(
    numeric_predictors(train_rows, selected_feature_columns),
    train_rows[TARGET_COLUMNS],
)
test_predictions = validate_predictions(
    final_model.predict(numeric_predictors(test_rows, selected_feature_columns)),
    expected_rows=len(test_rows),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
cv_results_records = []
for row in cv_results.to_dict(orient="records"):
    cv_results_records.append(
        {
            key: (
                str(value)
                if key == "subset"
                else int(value)
                if key == "feature_count"
                else float(value)
            )
            for key, value in row.items()
        }
    )
model_manifest = {
    "schema_version": "1.1",
    "model_path": str(MODEL_PATH),
    "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    "model_type": "sklearn.pipeline.Pipeline",
    "estimator": "Ridge",
    "preprocessor": "StandardScaler",
    "station_id": station_id,
    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
    "full_feature_columns": FULL_FEATURE_COLUMNS,
    "selected_subset": selected_subset,
    "feature_subset": selected_subset,
    "selected_feature_columns": selected_feature_columns,
    "feature_subsets": FEATURE_SUBSETS,
    "target_columns": TARGET_COLUMNS,
    "selected_alpha": float(selected_alpha),
    "selection_metric": CV_SELECTION_METRIC,
    "tie_breaking": [
        f"lowest aggregate CV {CV_SELECTION_METRIC.upper()}",
        "fewer features",
        "smaller alpha",
        "stable subset name",
    ],
    "cv_results": cv_results_records,
    "common_cohort_eligibility": {
        "contract": "full_feature_columns",
        "rule": "target_valid and complete full predictor and target contract",
        "train_raw_rows": len(train_features),
        "train_eligible_rows": len(train_rows),
        "test_raw_rows": len(test_features),
        "test_eligible_rows": len(test_rows),
        "same_folds_for_all_candidates": True,
    },
    "training": {
        "source_artifact": str(train_path),
        "raw_rows": len(train_features),
        "eligible_rows": len(train_rows),
        "eligibility": "target_valid and complete full predictor and target contract",
        "timestamp_start": train_rows["timestamp"].iloc[0].isoformat(),
        "timestamp_end": train_rows["timestamp"].iloc[-1].isoformat(),
    },
}
MODEL_METADATA_PATH.write_text(
    json.dumps(model_manifest, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved Ridge model to {MODEL_PATH}")
print(f"Saved Ridge model manifest to {MODEL_METADATA_PATH}")

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE, the same metrics for each lead in the direct 24-hour forecast, and a short preview for comparison with actual targets. Plot and MLflow labels identify both the selected subset and alpha. There is no second pass and no refitting.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
if not np.isfinite(aggregate_metrics[["mae", "rmse"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=f"ridge_test_{selected_subset}_alpha_{selected_alpha:g}",
    nested=False,
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "subset": selected_subset,
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "subset": selected_subset,
            "feature_count": len(selected_feature_columns),
            "feature_columns": json.dumps(selected_feature_columns),
            **INPUT_PARQUET_SHA256_PARAMS,
            "alpha": selected_alpha,
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "cv_selected_metric": float(
                cv_results.loc[
                    cv_results["subset"].eq(selected_subset)
                    & cv_results["alpha"].eq(selected_alpha),
                    f"{CV_SELECTION_METRIC}_mean",
                ].iloc[0]
            ),
            "scored_issue_times": len(test_rows),
        }
    )
    mlflow.log_metrics(
        {
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            **{
                f"test_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                for row in per_horizon_metrics.itertuples()
            },
        }
    )
    cv_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    cv_boxplot_values, horizon_labels = cv_error_boxplot_payload(
        cv_horizon_metrics,
        target_columns=TARGET_COLUMNS,
    )
    cv_rmse_mae_boxplots_fig = error_boxplots_figure(
        cv_boxplot_values,
        horizon_labels,
        title=f"Ridge CV errors — {selected_subset}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    (
        test_boxplot_values,
        test_horizon_labels,
        test_summary_markers,
    ) = absolute_error_boxplot_payload(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
    )
    test_error_boxplots_fig = error_boxplots_figure(
        test_boxplot_values,
        test_horizon_labels,
        title=f"Ridge final-test errors — {selected_subset}, alpha={selected_alpha:g}",
        x_axis_label="Forecast horizon",
        summary_markers=test_summary_markers,
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=f"Ridge predicted vs actual — {selected_subset}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
print(
    f"Ridge test results for {station_id} "
    f"(selected subset={selected_subset!r}, alpha={selected_alpha:g})"
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(
        test_rows,
        test_predictions,
        target_columns=TARGET_COLUMNS,
    ).head(PREDICTION_PREVIEW_ROWS)
)